### Extracting Text from Video

This process involves:
1.  **Extracting Audio**: Separating the audio track from the video file.
2.  **Speech-to-Text**: Transcribing the extracted audio into written text.

We will use `moviepy` for audio extraction and `SpeechRecognition` with the Google Web Speech API for transcription. Please note that the Google Web Speech API has usage limitations, and for more robust or large-scale applications, you might consider other speech-to-text services (e.g., Google Cloud Speech-to-Text, AWS Transcribe, AssemblyAI, OpenAI Whisper).

In [1]:
# Install necessary libraries
!pip install moviepy SpeechRecognition pydub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 32.9/32.9 MB 47.9 MB/s eta 0:00:00


In [2]:
import moviepy.editor as mp
import speech_recognition as sr
from pydub import AudioSegment
from pydub.silence import split_on_silence
import os

def extract_text_from_video(video_path, output_audio_path="extracted_audio.wav", chunk_folder="audio_chunks"): # noqa
    """
    Extracts text from the audio track of a video file.

    Args:
        video_path (str): The path to the input video file.
        output_audio_path (str): The path to save the extracted audio.
        chunk_folder (str): Folder to temporarily store audio chunks for transcription.

    Returns:
        str: The full transcribed text from the video.
    """
    print(f"[INFO] Extracting audio from {video_path}...")
    # 1. Extract audio from video
    try:
        video = mp.VideoFileClip(video_path)
        video.audio.write_audiofile(output_audio_path)
        print(f"[INFO] Audio extracted to {output_audio_path}")
    except Exception as e:
        print(f"[ERROR] Failed to extract audio: {e}")
        return ""

    # 2. Transcribe audio using SpeechRecognition
    r = sr.Recognizer()
    full_text = []

    # Load the audio file
    try:
        audio = AudioSegment.from_wav(output_audio_path)
    except Exception as e:
        print(f"[ERROR] Failed to load audio file: {e}")
        return ""

    # Split audio into chunks based on silence
    print("[INFO] Splitting audio into chunks...")
    chunks = split_on_silence(
        audio,
        min_silence_len=500,  # Minimum silence length in milliseconds
        silence_thresh=-40,   # Silence threshold (lower is more sensitive)
        keep_silence=200      # Keep 200 ms of silence around chunks
    )

    if not os.path.exists(chunk_folder):
        os.makedirs(chunk_folder)

    print(f"[INFO] Transcribing {len(chunks)} audio chunks...")
    for i, chunk in enumerate(chunks):
        chunk_filename = os.path.join(chunk_folder, f"chunk{i}.wav")
        chunk.export(chunk_filename, format="wav")
        with sr.AudioFile(chunk_filename) as source:
            audio_listened = r.record(source)
            try:
                text = r.recognize_google(audio_listened)
                full_text.append(text)
                print(f"[INFO] Chunk {i+1}/{len(chunks)} transcribed.")
            except sr.UnknownValueError:
                print(f"[WARNING] Could not understand audio in chunk {i+1}")
            except sr.RequestError as e:
                print(f"[ERROR] Could not request results from Google Web Speech API; {e} for chunk {i+1}") # noqa
        os.remove(chunk_filename)

    # Clean up temporary audio file and folder
    if os.path.exists(output_audio_path):
        os.remove(output_audio_path)
    if os.path.exists(chunk_folder):
        os.rmdir(chunk_folder)

    return " ".join(full_text)

# Example usage (you would need a video file uploaded to Colab, e.g., 'my_video.mp4')
# For demonstration, let's assume you have a video file named 'sample_video.mp4'
# If you want to test, upload a small video file to your Colab environment
# video_file = 'sample_video.mp4'
# if os.path.exists(video_file):
#     transcribed_text = extract_text_from_video(video_file)
#     print("\n--- Transcribed Text ---")
#     print(transcribed_text)
# else:
#     print(f"[INFO] Please upload a video file named '{video_file}' to test the function.")


/usr/local/lib/python3.12/dist-packages/moviepy/config_defaults.py:47: SyntaxWarning: invalid escape sequence '\P'
  IMAGEMAGICK_BINARY = r"C:\Program Files\ImageMagick-6.8.8-Q16\magick.exe"
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:294: SyntaxWarning: invalid escape sequence '\d'
  lines_video = [l for l in lines if ' Video: ' in l and re.search('\d+x\d+', l)]
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:367: SyntaxWarning: invalid escape sequence '\d'
  rotation_lines = [l for l in lines if 'rotate          :' in l and re.search('\d+$', l)]
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:370: SyntaxWarning: invalid escape sequence '\d'
  match = re.search('\d+$', rotation_line)
  if event.key is 'enter':

  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)

  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)

  elif re.match('(flt)p?( \(default\))?$', token):

  elif re.m

In [3]:
video_file = 'video.nikki.mp4'

if os.path.exists(video_file):
    print(f"[INFO] Processing video file: {video_file}")
    transcribed_text = extract_text_from_video(video_file)
    print("\n--- Transcribed Text ---")
    print(transcribed_text)
else:
    print(f"[ERROR] Video file '{video_file}' not found. Please upload it to your Colab environment or specify the correct path.")

[INFO] Processing video file: video.nikki.mp4
[INFO] Extracting audio from video.nikki.mp4...
MoviePy - Writing audio in extracted_audio.wav


MoviePy - Done.
[INFO] Audio extracted to extracted_audio.wav
[INFO] Splitting audio into chunks...


[INFO] Transcribing 1 audio chunks...
[WARNING] Could not understand audio in chunk 1

--- Transcribed Text ---

